# Data Exploration: Synthetic Time Series with Structural Transitions

This notebook visualizes the generated synthetic time series data and explores the different dynamics and concept drift scenarios.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (15, 8)

## Load Data

In [ ]:
# Load fold bifurcation data
fold_ts = np.load('../data/raw/time_series_fold.npy')
fold_labels = np.load('../data/raw/labels_fold.npy')

# Load saddle-node bifurcation data
saddle_ts = np.load('../data/raw/time_series_saddle_node.npy')
saddle_labels = np.load('../data/raw/labels_saddle_node.npy')

# Load Hopf bifurcation data
hopf_ts = np.load('../data/raw/time_series_hopf.npy')
hopf_labels = np.load('../data/raw/labels_hopf.npy')

print(f"Fold bifurcation data shape: {fold_ts.shape}")
print(f"Saddle-node bifurcation data shape: {saddle_ts.shape}")
print(f"Hopf bifurcation data shape: {hopf_ts.shape}")

## Visualize Different Dynamics Types

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(15, 12))

# Select a random realization from each type
idx = 0

# Fold bifurcation
axes[0].plot(fold_ts[idx], linewidth=0.8, color='steelblue')
axes[0].axvspan(0, 1000, alpha=0.2, color='green', label='Stable')
axes[0].axvspan(1000, 1800, alpha=0.2, color='orange', label='Approaching Transition')
axes[0].axvspan(1800, 2500, alpha=0.2, color='red', label='Post-Transition')
axes[0].set_title('Fold Bifurcation Dynamics', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Value', fontsize=12)
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.3)

# Saddle-node bifurcation
axes[1].plot(saddle_ts[idx], linewidth=0.8, color='darkgreen')
axes[1].axvspan(0, 1000, alpha=0.2, color='green')
axes[1].axvspan(1000, 1800, alpha=0.2, color='orange')
axes[1].axvspan(1800, 2500, alpha=0.2, color='red')
axes[1].set_title('Saddle-Node Bifurcation Dynamics', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Value', fontsize=12)
axes[1].grid(True, alpha=0.3)

# Hopf bifurcation
axes[2].plot(hopf_ts[idx], linewidth=0.8, color='purple')
axes[2].axvspan(0, 1000, alpha=0.2, color='green')
axes[2].axvspan(1000, 1800, alpha=0.2, color='orange')
axes[2].axvspan(1800, 2500, alpha=0.2, color='red')
axes[2].set_title('Hopf Bifurcation Dynamics (Oscillatory Transition)', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Time Step', fontsize=12)
axes[2].set_ylabel('Value', fontsize=12)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/dynamics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Statistical Properties Across Regimes

In [ ]:
def compute_regime_statistics(time_series, labels):
    """
    Compute statistics for each regime.
    """
    stats = {}
    
    for regime in [0, 1, 2]:
        regime_data = time_series[:, labels[0] == regime]
        stats[regime] = {
            'mean': np.mean(regime_data),
            'std': np.std(regime_data),
            'variance': np.var(regime_data),
            'autocorr_lag1': np.corrcoef(regime_data[:, :-1].flatten(), 
                                         regime_data[:, 1:].flatten())[0, 1]
        }
    
    return stats

# Compute statistics for fold bifurcation
fold_stats = compute_regime_statistics(fold_ts, fold_labels)

print("Fold Bifurcation Statistics:")
print("="*50)
for regime, stats in fold_stats.items():
    regime_names = ['Stable', 'Approaching Transition', 'Post-Transition']
    print(f"\n{regime_names[regime]}:")
    for key, value in stats.items():
        print(f"  {key}: {value:.4f}")

## Variance and Autocorrelation Evolution

In [ ]:
def rolling_statistics(time_series, window=50):
    """
    Compute rolling statistics.
    """
    n_steps = time_series.shape[1]
    rolling_var = np.zeros(n_steps - window + 1)
    rolling_autocorr = np.zeros(n_steps - window + 1)
    
    for i in range(n_steps - window + 1):
        window_data = time_series[:, i:i+window]
        rolling_var[i] = np.var(window_data)
        
        # Autocorrelation lag-1
        if window > 1:
            autocorr = np.corrcoef(window_data[:, :-1].flatten(), 
                                   window_data[:, 1:].flatten())[0, 1]
            rolling_autocorr[i] = autocorr
    
    return rolling_var, rolling_autocorr

# Compute rolling statistics for fold bifurcation
fold_var, fold_autocorr = rolling_statistics(fold_ts, window=100)

fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Rolling variance
axes[0].plot(fold_var, linewidth=1.5, color='darkred')
axes[0].axvspan(0, 1000, alpha=0.2, color='green')
axes[0].axvspan(1000, 1800, alpha=0.2, color='orange')
axes[0].axvspan(1800, len(fold_var), alpha=0.2, color='red')
axes[0].set_title('Rolling Variance (Early Warning Signal)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Variance', fontsize=12)
axes[0].grid(True, alpha=0.3)

# Rolling autocorrelation
axes[1].plot(fold_autocorr, linewidth=1.5, color='darkblue')
axes[1].axvspan(0, 1000, alpha=0.2, color='green')
axes[1].axvspan(1000, 1800, alpha=0.2, color='orange')
axes[1].axvspan(1800, len(fold_autocorr), alpha=0.2, color='red')
axes[1].set_title('Rolling Autocorrelation (Critical Slowing Down)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Time Step', fontsize=12)
axes[1].set_ylabel('Autocorrelation (lag-1)', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/early_warning_signals.png', dpi=150, bbox_inches='tight')
plt.show()

## Concept Drift Scenarios

In [ ]:
# Load drift scenarios
drift_noise = np.load('../data/drift_scenarios/time_series_fold_noise_variance.npy')
drift_mean = np.load('../data/drift_scenarios/time_series_fold_mean_shift.npy')
drift_scale = np.load('../data/drift_scenarios/time_series_fold_scale.npy')

fig, axes = plt.subplots(3, 1, figsize=(15, 12))

idx = 0

# Noise variance drift
axes[0].plot(drift_noise[idx], linewidth=0.8, color='steelblue')
axes[0].axvspan(500, 1000, alpha=0.3, color='yellow', label='Drift Period')
axes[0].set_title('Concept Drift: Noise Variance Change', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Value', fontsize=12)
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.3)

# Mean shift drift
axes[1].plot(drift_mean[idx], linewidth=0.8, color='darkgreen')
axes[1].axvspan(500, 1000, alpha=0.3, color='yellow')
axes[1].set_title('Concept Drift: Mean Shift', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Value', fontsize=12)
axes[1].grid(True, alpha=0.3)

# Scale drift
axes[2].plot(drift_scale[idx], linewidth=0.8, color='purple')
axes[2].axvspan(500, 1000, alpha=0.3, color='yellow')
axes[2].set_title('Concept Drift: Scale Change', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Time Step', fontsize=12)
axes[2].set_ylabel('Value', fontsize=12)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/concept_drift_scenarios.png', dpi=150, bbox_inches='tight')
plt.show()

## Multiple Realizations Comparison

In [ ]:
# Plot multiple realizations to show variability
fig, ax = plt.subplots(figsize=(15, 6))

n_plot = 10
for i in range(n_plot):
    ax.plot(fold_ts[i], alpha=0.4, linewidth=0.8)

ax.axvspan(0, 1000, alpha=0.15, color='green', label='Stable')
ax.axvspan(1000, 1800, alpha=0.15, color='orange', label='Approaching Transition')
ax.axvspan(1800, 2500, alpha=0.15, color='red', label='Post-Transition')
ax.set_title(f'{n_plot} Realizations of Fold Bifurcation Dynamics', fontsize=14, fontweight='bold')
ax.set_xlabel('Time Step', fontsize=12)
ax.set_ylabel('Value', fontsize=12)
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/multiple_realizations.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

This notebook demonstrates:

1. **Three types of bifurcation dynamics**: Fold, Saddle-Node, and Hopf
2. **Early warning signals**: Increasing variance and autocorrelation before transitions
3. **Concept drift scenarios**: Noise variance, mean shift, and scale changes
4. **Variability across realizations**: Multiple stochastic realizations of the same dynamics

The data is ready for preprocessing and model training.